# Dataset Genration for Q1

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import lit
from pyspark.sql.types import *

spark = SparkSession.builder.getOrCreate()

transactions_data = [
    ("t1", "u1", "p1", "$120.50", "USD", "2025-02-01 10:15:00"),
    ("t2", "u2", "p2", "$200.00", "USD", "01-02-2025 11:20:00"),  
    ("t3", None, "p3", "$50.00", "USD", "2025/02/01 12:00:00"), 
    ("t4", "u3", "p4", "-$30.00", "USD", "2025-02-01 13:00:00"), 
    ("t5", "u1", "p5", "$75.00", "USD", "2025-02-01 14:00:00"),
    ("t5", "u1", "p5", "$75.00", "USD", "2025-02-01 14:05:00"), 
    ("t6", "u4", "p2", "$500.00", "USD", "2025-02-02 09:00:00"),
    ("t7", "u2", "p3", "$300.00", "USD", "2025-02-02 10:30:00"),
    ("t8", "u3", "p1", "$150.00", "USD", "2025-02-02 11:00:00"),
]

transactions_schema = StructType([
    StructField("transaction_id", StringType(), True),
    StructField("user_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("amount", StringType(), True),
    StructField("currency", StringType(), True),
    StructField("transaction_timestamp", StringType(), True),
])

transactions_df = spark.createDataFrame(transactions_data, schema=transactions_schema)

users_data = [
    ("u1", "uk", "2024-01-01"),
    ("u2", "Us", "2024-02-01"),
    ("u3", "IN", "2024-03-01"),
    ("u3", "in", "2024-03-02"),  
    ("u4", None, "2024-04-01"),  
]

users_schema = StructType([
    StructField("user_id", StringType(), True),
    StructField("country", StringType(), True),
    StructField("signup_date", StringType(), True),
])

users_df = spark.createDataFrame(users_data, schema=users_schema)




In [0]:
#CHANGE BELOW LOCATION TO SAVE THIS DATA 
target = "/Volumes/pyspark/bronze/raw_ingestion/csv"

#####
transactions_df.write.mode("overwrite").option("header", True).csv(f"{target}/transactions_csv")

users_df.write.mode("overwrite").option("header", True).csv(f"{target}/users_csv")


# Pair Programming Task — End-to-End PySpark Pipeline
---
## Scenario

You are given two datasets:

### 📁 `transactions.csv`

```
transaction_id (string)
user_id (string)
product_id (string)
amount (string)
currency (string)
transaction_timestamp (string)
```

Problems:

* amount contains currency symbols (e.g. "$120.50")
* timestamp format inconsistent
* duplicates exist
* some null user_id rows
* some negative amounts

---

### 📁 `users.csv`

```
user_id (string)
country (string)
signup_date (string)
```

Problems:

* duplicate users
* country casing inconsistent
* null values

---




## Step 1 — Read Data

* Read both datasets
* Define schema explicitly
* Handle malformed rows safely

---

## Step 2 — Clean Data

### Transactions:

* Remove null user_id
* Remove negative amounts
* Clean amount → cast to double
* Standardise timestamp → proper timestamp type
* Deduplicate based on latest transaction_timestamp per transaction_id



### Users:

* Deduplicate based on latest signup_date
* Standardise country to uppercase

---

## Step 3 — Transform

1. Join transactions with users
2. Calculate:

   * Daily revenue per country
3. Return:

   * transaction_date
   * country
   * total_revenue

---


## Step 4 — Advanced Requirement

For each country and day:

* Return top 3 users by revenue

Output:

```
transaction_date
country
user_id
total_user_revenue
rank
```

---


## Step 5 — Performance Discussion

After coding they will ask:

* What causes shuffle here?
* Where would you broadcast?
* How would you optimise for 1 billion rows?
* How would you partition when writing?
* How would you productionise this?

---